# Azurity Pharmaceuticals - Prescription Volume Forecasting

This notebook demonstrates an end-to-end ML workflow for pharmaceutical demand forecasting:

1. **Feature Store** - Point-in-time correct features for prescription forecasting
2. **Model Training** - XGBoost regressor for 30-day volume prediction
3. **Model Registry** - Register and version the trained model
4. **Streamlit Visualization** - Interactive A/B test analysis

**Business Value**: Predict prescription volume by product/HCP to optimize inventory, sales rep targeting, and promotional campaigns.

## 1. Setup and Imports

In [ ]:
%matplotlib inline
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
print(f"Connected to Snowflake: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

In [ ]:
session.sql("USE DATABASE AZURITY_DEMO_DB").collect()
session.sql("USE SCHEMA ML").collect()
session.sql("USE WAREHOUSE AZURITY_ML_WH").collect()
print("Context set to AZURITY_DEMO_DB.ML")

## 2. Explore Azurity Data

In [ ]:
print("=" * 60)
print("AZURITY PRODUCT PORTFOLIO")
print("=" * 60)
products_df = session.table("RAW.PRODUCTS").to_pandas()
display(products_df)

In [ ]:
print("=" * 60)
print("PRESCRIPTION DATA SUMMARY")
print("=" * 60)

rx_summary = session.sql("""
    SELECT 
        p.PRODUCT_NAME,
        COUNT(DISTINCT r.HCP_ID) AS UNIQUE_HCPS,
        COUNT(*) AS TOTAL_RX,
        SUM(r.QUANTITY) AS TOTAL_UNITS,
        MIN(r.FILL_DATE) AS FIRST_RX,
        MAX(r.FILL_DATE) AS LAST_RX
    FROM RAW.PRESCRIPTIONS r
    JOIN RAW.PRODUCTS p ON r.PRODUCT_ID = p.PRODUCT_ID
    GROUP BY p.PRODUCT_NAME
    ORDER BY TOTAL_UNITS DESC
""").to_pandas()
display(rx_summary)

In [ ]:
print("=" * 60)
print("HEALTHCARE PROVIDER DISTRIBUTION BY DECILE")
print("=" * 60)

hcp_decile = session.sql("""
    SELECT 
        DECILE,
        COUNT(*) AS HCP_COUNT,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS PCT
    FROM RAW.HEALTHCARE_PROVIDERS
    GROUP BY DECILE
    ORDER BY DECILE DESC
""").to_pandas()
print(hcp_decile.to_string(index=False))

## 3. Feature Store Setup

The Snowflake Feature Store provides:
- **Centralized feature management** - Single source of truth for ML features
- **Point-in-time correctness** - ASOF JOIN prevents data leakage in backtesting
- **Automatic refresh** - Features stay up-to-date as source data changes

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database="AZURITY_DEMO_DB",
    name="ML",
    default_warehouse="AZURITY_ML_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)
print("Feature Store initialized")

In [ ]:
product_entity = Entity(name="PRODUCT", join_keys=["PRODUCT_ID"])
hcp_entity = Entity(name="HCP", join_keys=["HCP_ID"])

try:
    fs.register_entity(product_entity)
    print("Registered entity: PRODUCT")
except Exception as e:
    print(f"Entity PRODUCT already exists or error: {e}")

try:
    fs.register_entity(hcp_entity)
    print("Registered entity: HCP")
except Exception as e:
    print(f"Entity HCP already exists or error: {e}")

print("\nRegistered Entities:")
for entity in fs.list_entities().collect():
    print(f"  - {entity['NAME']}")

In [ ]:
rx_features_query = """
SELECT 
    PRODUCT_ID,
    HCP_ID,
    FILL_DATE AS AS_OF_DATE,
    REGION,
    SUM(QUANTITY) OVER (
        PARTITION BY PRODUCT_ID, HCP_ID 
        ORDER BY FILL_DATE 
        ROWS BETWEEN 13 PRECEDING AND CURRENT ROW
    ) AS RX_VOLUME_2W,
    SUM(QUANTITY) OVER (
        PARTITION BY PRODUCT_ID, HCP_ID 
        ORDER BY FILL_DATE 
        ROWS BETWEEN 27 PRECEDING AND CURRENT ROW
    ) AS RX_VOLUME_4W,
    AVG(QUANTITY) OVER (
        PARTITION BY PRODUCT_ID, HCP_ID 
        ORDER BY FILL_DATE 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS RX_AVG_LAST_WEEK,
    COUNT(*) OVER (
        PARTITION BY PRODUCT_ID, HCP_ID 
        ORDER BY FILL_DATE 
        ROWS BETWEEN 27 PRECEDING AND CURRENT ROW
    ) AS RX_COUNT_4W
FROM RAW.PRESCRIPTIONS
"""

rx_features_df = session.sql(rx_features_query)
print("Created prescription features DataFrame")
rx_features_df.limit(5).show()

In [ ]:
try:
    rx_feature_view = fs.get_feature_view("RX_FEATURES", "v1")
    print("Feature view RX_FEATURES already exists, using existing version")
except:
    rx_feature_view = FeatureView(
        name="RX_FEATURES",
        entities=[product_entity, hcp_entity],
        feature_df=rx_features_df,
        timestamp_col="AS_OF_DATE",
        refresh_freq="1 day",
        desc="Prescription volume features for demand forecasting"
    )
    rx_feature_view = fs.register_feature_view(
        feature_view=rx_feature_view,
        version="v1",
        block=True
    )
    print("Registered Feature View: RX_FEATURES v1")

print("\nFeature View Details:")
print(f"  Name: {rx_feature_view.name}")
print(f"  Version: {rx_feature_view.version}")

## 4. Generate Training Dataset with Point-in-Time Correctness

We use the Feature Store to generate training data that is **point-in-time correct**:
- For each historical date, we only use features that were available at that time
- This prevents data leakage and ensures backtesting accuracy

In [ ]:
spine_query = """
SELECT DISTINCT
    r.PRODUCT_ID,
    r.HCP_ID,
    r.FILL_DATE AS AS_OF_DATE,
    COALESCE(SUM(r2.QUANTITY), 0) AS TARGET_NEXT_30_DAY_VOLUME
FROM RAW.PRESCRIPTIONS r
LEFT JOIN RAW.PRESCRIPTIONS r2 
    ON r.PRODUCT_ID = r2.PRODUCT_ID 
    AND r.HCP_ID = r2.HCP_ID
    AND r2.FILL_DATE > r.FILL_DATE 
    AND r2.FILL_DATE <= DATEADD('day', 30, r.FILL_DATE)
WHERE r.FILL_DATE BETWEEN '2024-06-01' AND '2025-12-01'
GROUP BY r.PRODUCT_ID, r.HCP_ID, r.FILL_DATE
"""

spine_df = session.sql(spine_query)
print(f"Spine DataFrame created")
spine_df.limit(5).show()

In [ ]:
training_data = fs.generate_dataset(
    name="rx_training_data",
    spine_df=spine_df,
    features=[rx_feature_view],
    spine_timestamp_col="AS_OF_DATE",
    spine_label_cols=["TARGET_NEXT_30_DAY_VOLUME"],
    include_feature_view_timestamp_col=False
).read.to_snowpark_dataframe()

print("Training data generated with point-in-time correctness")
print(f"Columns: {training_data.columns}")
training_data.limit(5).show()

In [ ]:
training_pdf = training_data.to_pandas()
training_pdf = training_pdf.dropna()

print(f"Training samples: {len(training_pdf):,}")
print(f"Target mean: {training_pdf['TARGET_NEXT_30_DAY_VOLUME'].mean():.2f}")
print(f"Target std: {training_pdf['TARGET_NEXT_30_DAY_VOLUME'].std():.2f}")

## 5. Train XGBoost Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

le_product = LabelEncoder()
le_region = LabelEncoder()

training_pdf['PRODUCT_ENCODED'] = le_product.fit_transform(training_pdf['PRODUCT_ID'])
training_pdf['REGION_ENCODED'] = le_region.fit_transform(training_pdf['REGION'].fillna('Unknown'))

feature_cols = [
    'PRODUCT_ENCODED',
    'REGION_ENCODED',
    'RX_VOLUME_2W',
    'RX_VOLUME_4W',
    'RX_AVG_LAST_WEEK',
    'RX_COUNT_4W'
]

X = training_pdf[feature_cols].fillna(0)
y = training_pdf['TARGET_NEXT_30_DAY_VOLUME']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("XGBoost model trained successfully!")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=" * 60)
print("MODEL PERFORMANCE METRICS")
print("=" * 60)
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")

In [ ]:
print("=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

importance_df['Importance'] = (importance_df['Importance'] * 100).round(1).astype(str) + '%'
print(importance_df.to_string(index=False))

## 6. Register Model in Snowflake Model Registry

The Model Registry provides:
- **Version control** - Track model iterations
- **Metadata management** - Store metrics, comments, tags
- **SQL inference** - Call model.predict() directly from SQL
- **RBAC** - Control access to models with Snowflake roles

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name="AZURITY_DEMO_DB", schema_name="ML")

model_name = "AZURITY_RX_FORECASTER"
version_name = "V1"

try:
    registry.delete_model(model_name)
    print(f"Deleted existing model {model_name}")
except:
    pass

mv = registry.log_model(
    model,
    model_name=model_name,
    version_name=version_name,
    sample_input_data=X_train.head(100),
    target_platforms=["WAREHOUSE"],
    metrics={
        "rmse": float(rmse),
        "mae": float(mae),
        "r2": float(r2),
        "training_samples": len(X_train),
        "test_samples": len(X_test)
    },
    comment="Prescription volume forecasting model for Azurity products. Predicts 30-day volume by product/HCP."
)

print("=" * 60)
print("MODEL REGISTERED SUCCESSFULLY")
print("=" * 60)
print(f"Model: AZURITY_DEMO_DB.ML.{model_name}")
print(f"Version: {version_name}")
print(f"\nMetrics:")
for k, v in mv.show_metrics().items():
    print(f"  {k}: {v}")

In [ ]:
print("=" * 60)
print("MODELS IN REGISTRY")
print("=" * 60)
models_df = registry.show_models()
display(models_df[['name', 'comment', 'default_version_name']])

## 7. SQL Inference Demo

Once registered, the model can be called directly from SQL - no Python required!

In [ ]:
inference_result = mv.run(X_test.head(10), function_name="predict")

print("=" * 60)
print("MODEL INFERENCE RESULTS")
print("=" * 60)

comparison = X_test.head(10).copy()
comparison['ACTUAL'] = y_test.head(10).values
comparison['PREDICTED'] = inference_result['output_feature_0'].values
comparison['ERROR'] = abs(comparison['ACTUAL'] - comparison['PREDICTED'])

display(comparison[['RX_VOLUME_2W', 'RX_VOLUME_4W', 'ACTUAL', 'PREDICTED', 'ERROR']])

In [ ]:
print("=" * 60)
print("SQL INFERENCE EXAMPLE")
print("=" * 60)
print("""
-- Run this SQL to score new data:
SELECT 
    PRODUCT_ID,
    HCP_ID,
    AZURITY_RX_FORECASTER!PREDICT(
        PRODUCT_ENCODED,
        REGION_ENCODED,
        RX_VOLUME_2W,
        RX_VOLUME_4W,
        RX_AVG_LAST_WEEK,
        RX_COUNT_4W
    ) AS PREDICTED_30_DAY_VOLUME
FROM AZURITY_DEMO_DB.ML.SCORING_INPUT_VIEW;
""")

## 8. Data Visualization

Visualize campaign performance and product trends. For interactive dashboards, see the standalone Streamlit app (`streamlit_app.py`).

In [ ]:
import matplotlib.pyplot as plt

print("=" * 60)
print("EMAIL CAMPAIGN A/B TEST ANALYSIS")
print("=" * 60)

campaign_results = session.sql("""
    SELECT 
        VARIANT,
        COUNT(*) AS EMAILS_SENT,
        SUM(CASE WHEN OPEN_DATE IS NOT NULL THEN 1 ELSE 0 END) AS OPENS,
        SUM(CASE WHEN CLICK_DATE IS NOT NULL THEN 1 ELSE 0 END) AS CLICKS,
        ROUND(SUM(CASE WHEN OPEN_DATE IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS OPEN_RATE,
        ROUND(SUM(CASE WHEN CLICK_DATE IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / 
              NULLIF(SUM(CASE WHEN OPEN_DATE IS NOT NULL THEN 1 ELSE 0 END), 0), 2) AS CTR
    FROM RAW.EMAIL_CAMPAIGNS
    GROUP BY VARIANT
    ORDER BY VARIANT
""").to_pandas()

fig, ax = plt.subplots(figsize=(10, 5))
campaign_results.set_index('VARIANT')[['OPENS', 'CLICKS']].plot(kind='bar', ax=ax)
ax.set_title("Email Campaign A/B Test: Engagement by Variant")
ax.set_ylabel("Count")
ax.set_xlabel("Variant")
plt.xticks(rotation=0)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

print("\nCampaign Metrics:")
display(campaign_results)

In [ ]:
import matplotlib.pyplot as plt

print("=" * 60)
print("PRODUCT PERFORMANCE TRENDS")
print("=" * 60)

product_trend = session.sql("""
    SELECT 
        p.PRODUCT_NAME,
        DATE_TRUNC('month', r.FILL_DATE)::DATE AS MONTH,
        SUM(r.QUANTITY) AS MONTHLY_VOLUME,
        COUNT(DISTINCT r.HCP_ID) AS UNIQUE_PRESCRIBERS
    FROM RAW.PRESCRIPTIONS r
    JOIN RAW.PRODUCTS p ON r.PRODUCT_ID = p.PRODUCT_ID
    GROUP BY 1, 2
    ORDER BY 1, 2
""").to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for product in product_trend['PRODUCT_NAME'].unique():
    product_data = product_trend[product_trend['PRODUCT_NAME'] == product]
    axes[0].plot(product_data['MONTH'], product_data['MONTHLY_VOLUME'], label=product, marker='o', markersize=3)
    axes[1].plot(product_data['MONTH'], product_data['UNIQUE_PRESCRIBERS'], label=product, marker='o', markersize=3)

axes[0].set_title("Monthly Prescription Volume by Product")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Volume")
axes[0].legend(loc='upper left', fontsize=8)
axes[0].tick_params(axis='x', rotation=45)

axes[1].set_title("Monthly Unique Prescribers by Product")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Unique HCPs")
axes[1].legend(loc='upper left', fontsize=8)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nTop Products by Total Volume:")
summary = product_trend.groupby('PRODUCT_NAME').agg({
    'MONTHLY_VOLUME': 'sum',
    'UNIQUE_PRESCRIBERS': 'mean'
}).round(0).sort_values('MONTHLY_VOLUME', ascending=False)
summary.columns = ['TOTAL_VOLUME', 'AVG_MONTHLY_PRESCRIBERS']
display(summary)

## 9. Summary

This notebook demonstrated:

| Capability | What We Showed |
|------------|----------------|
| **Feature Store** | Created entities, registered feature views with auto-refresh |
| **Point-in-Time** | Generated training data with ASOF JOIN to prevent leakage |
| **Model Training** | XGBoost regressor for 30-day prescription volume |
| **Model Registry** | Registered model with metrics, versioning, SQL inference |
| **Streamlit** | Interactive A/B test analysis with dynamic date ranges |

### Next Steps
1. **MLOps**: Add data drift monitoring with Snowflake ML Observability
2. **Batch Scoring**: Schedule daily inference using Snowflake Tasks
3. **Model Fallback**: Implement automatic rollback on performance degradation

In [ ]:
print("=" * 60)
print("AZURITY ML DEMO COMPLETE")
print("=" * 60)
print(f"\nArtifacts Created:")
print(f"  - Feature Store: AZURITY_DEMO_DB.ML (schema)")
print(f"  - Feature View: RX_FEATURES v1")
print(f"  - Model: AZURITY_DEMO_DB.ML.AZURITY_RX_FORECASTER")
print(f"\nModel Performance:")
print(f"  - RMSE: {rmse:.2f}")
print(f"  - MAE: {mae:.2f}")
print(f"  - R2 Score: {r2:.4f}")

## EXTRA: Experiment Tracking - Train Model V2 with Different Hyperparameters

Use Snowflake ML Experiments to track training runs and compare model versions. This enables you to:
- Compare hyperparameter configurations
- Track metrics across training iterations
- Visualize model performance in Snowsight (AI & ML → Experiments)

In [ ]:
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.experiment.callback.xgboost import SnowflakeXgboostCallback
from snowflake.ml.model.model_signature import infer_signature

exp = ExperimentTracking(session=session)
exp.set_experiment("AZURITY_RX_FORECASTING_EXPERIMENT")

sig = infer_signature(X_train, y_train)

print("=" * 60)
print("EXPERIMENT TRACKING - MODEL V1 (Baseline)")
print("=" * 60)

callback_v1 = SnowflakeXgboostCallback(
    exp, 
    model_name="AZURITY_RX_FORECASTER",
    model_signature=sig
)

model_v1 = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    callbacks=[callback_v1]
)

with exp.start_run("V1_Baseline"):
    exp.log_params({
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "model_type": "XGBRegressor"
    })
    
    model_v1.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    y_pred_v1 = model_v1.predict(X_test)
    rmse_v1 = np.sqrt(mean_squared_error(y_test, y_pred_v1))
    mae_v1 = mean_absolute_error(y_test, y_pred_v1)
    r2_v1 = r2_score(y_test, y_pred_v1)
    
    exp.log_metrics({
        "rmse": float(rmse_v1),
        "mae": float(mae_v1),
        "r2": float(r2_v1)
    })
    
    print(f"V1 RMSE: {rmse_v1:.2f}")
    print(f"V1 MAE: {mae_v1:.2f}")
    print(f"V1 R²: {r2_v1:.4f}")

In [ ]:
print("=" * 60)
print("EXPERIMENT TRACKING - MODEL V2 (Tuned Hyperparameters)")
print("=" * 60)

callback_v2 = SnowflakeXgboostCallback(
    exp, 
    model_name="AZURITY_RX_FORECASTER",
    model_signature=sig
)

model_v2 = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    callbacks=[callback_v2]
)

with exp.start_run("V2_Tuned"):
    exp.log_params({
        "n_estimators": 200,
        "max_depth": 8,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "min_child_weight": 3,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "model_type": "XGBRegressor"
    })
    
    model_v2.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    y_pred_v2 = model_v2.predict(X_test)
    rmse_v2 = np.sqrt(mean_squared_error(y_test, y_pred_v2))
    mae_v2 = mean_absolute_error(y_test, y_pred_v2)
    r2_v2 = r2_score(y_test, y_pred_v2)
    
    exp.log_metrics({
        "rmse": float(rmse_v2),
        "mae": float(mae_v2),
        "r2": float(r2_v2)
    })
    
    print(f"V2 RMSE: {rmse_v2:.2f}")
    print(f"V2 MAE: {mae_v2:.2f}")
    print(f"V2 R²: {r2_v2:.4f}")

print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(f"{'Metric':<10} {'V1 (Baseline)':<15} {'V2 (Tuned)':<15} {'Improvement':<15}")
print("-" * 55)
rmse_imp = ((rmse_v1 - rmse_v2) / rmse_v1) * 100
mae_imp = ((mae_v1 - mae_v2) / mae_v1) * 100
r2_imp = ((r2_v2 - r2_v1) / abs(r2_v1)) * 100
print(f"{'RMSE':<10} {rmse_v1:<15.2f} {rmse_v2:<15.2f} {rmse_imp:>+.1f}%")
print(f"{'MAE':<10} {mae_v1:<15.2f} {mae_v2:<15.2f} {mae_imp:>+.1f}%")
print(f"{'R²':<10} {r2_v1:<15.4f} {r2_v2:<15.4f} {r2_imp:>+.1f}%")

In [ ]:
best_model = model_v2 if rmse_v2 < rmse_v1 else model_v1
best_version = "V2" if rmse_v2 < rmse_v1 else "V1"
best_rmse = min(rmse_v1, rmse_v2)

mv_v2 = registry.log_model(
    model_v2,
    model_name=model_name,
    version_name="V2",
    sample_input_data=X_train.head(100),
    target_platforms=["WAREHOUSE"],
    metrics={
        "rmse": float(rmse_v2),
        "mae": float(mae_v2),
        "r2": float(r2_v2),
        "training_samples": len(X_train),
        "test_samples": len(X_test)
    },
    comment="Tuned prescription forecasting model with deeper trees and regularization."
)

print("=" * 60)
print("MODEL V2 REGISTERED TO SNOWFLAKE MODEL REGISTRY")
print("=" * 60)
print(f"Model: AZURITY_DEMO_DB.ML.{model_name}")
print(f"Version: V2")
print(f"\nBest performing model: {best_version} (RMSE: {best_rmse:.2f})")
print(f"\n💡 View experiment comparison in Snowsight:")
print(f"   Navigate to: AI & ML → Experiments → AZURITY_RX_FORECASTING_EXPERIMENT")